In [82]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [83]:
movies = pd.read_csv('../../datos/movie_recomendations/movies_cleaned.csv')
movies

,movieId,title,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year
0,1,Toy Story (1995),0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1995.0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1995.0
2,3,Grumpier Old Men (1995),0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,1995.0
3,4,Waiting to Exhale (1995),0,0,0,0,1,0,0,1,...,0,0,0,0,1,0,0,0,0,1995.0
4,5,Father of the Bride Part II (1995),0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1995.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57233,209155,Santosh Subramaniam (2008),1,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,2008.0
57234,209157,We (2018),0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,2018.0
57235,209159,Window of the Soul (2001),0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,2001.0
57236,209163,Bad Poems (2018),0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,2018.0


In [84]:
movies = movies.sample(5000)
movies.reset_index(inplace=True)
movies.drop(columns=['index'], inplace=True)
movies

,movieId,title,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year
0,71490,Grace (2009),0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,1,0,0,2009.0
1,190627,On the Yeti Trail (2014),0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,2014.0
2,163563,Can We Take a Joke? (2015),0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,2015.0
3,121241,Sweeney Todd: The Demon Barber of Fleet Street...,0,0,0,0,0,0,0,1,...,1,0,1,0,0,0,1,0,0,1982.0
4,5433,Silver Bullet (Stephen King's Silver Bullet) (...,0,1,0,0,0,0,0,1,...,1,0,0,1,0,0,1,0,0,1985.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,47940,Time to Leave (2005),0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,2005.0
4996,111649,"Message to Garcia, A (1936)",0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,1936.0
4997,158815,Chasing Leprechauns (2013),0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,2013.0
4998,4147,Nico and Dani (Krámpack) (2000),0,0,0,0,1,0,0,1,...,0,0,0,0,1,0,0,0,0,2000.0


In [85]:
movie_test = movies['title'][0]
movie_test

'Grace (2009)'

In [100]:
def recommend_movies(movie_title, df, similarity_matrix, top_n=5):
    # Encontrar el índice de la película dada
    idx = df[df['title'] == movie_title].index[0]
    
    # Obtener similitudes con otras películas
    sim_scores = list(enumerate(similarity_matrix[idx]))
    
    # Ordenar por mayor similitud (excluyendo la propia película)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    
    # Obtener los títulos recomendados
    recommended_indices = [i[0] for i in sim_scores]
    return df.loc[recommended_indices, ['title', 'year']]
    #return df.loc[recommended_indices, ['title']]

In [101]:
# Normalizar el año
scaler = MinMaxScaler()
movies['year_scaled'] = scaler.fit_transform(movies[['year']])

In [102]:
genre_columns = movies.columns[2:-1]
content_features = movies[genre_columns].copy()
content_features['year'] = movies['year_scaled']

# Matriz de similitud (generos)
similarity_matrix = cosine_similarity(content_features)

In [103]:
content_features

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year
0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0.922481
1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0.961240
2,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0.968992
3,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,1,0,0,0.713178
4,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0.736434
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.891473
4996,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0.356589
4997,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0.953488
4998,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0.852713


In [104]:
similarity_matrix

array([[1.        , 0.32576433, 0.32712189, ..., 0.26278782, 0.4715832 ,
        0.26425105],
       [0.32576433, 1.        , 0.99999195, ..., 0.38740431, 0.30608896,
        0.38956141],
       [0.32712189, 0.99999195, 1.        , ..., 0.38901875, 0.30736453,
        0.39118483],
       ...,
       [0.26278782, 0.38740431, 0.38901875, ..., 1.        , 0.85429681,
        0.99999294],
       [0.4715832 , 0.30608896, 0.30736453, ..., 0.85429681, 1.        ,
        0.85412809],
       [0.26425105, 0.38956141, 0.39118483, ..., 0.99999294, 0.85412809,
        1.        ]])

In [105]:
year_filtrado = 2010
recomendaciones = recommend_movies(movie_test, movies, similarity_matrix)
#recomendaciones = recommend_movies("Toy Story (1995)", movies, similarity_matrix)
recomendaciones_filtradas = recomendaciones[recomendaciones['year'] >= year_filtrado]

In [106]:
print(f'Recomendaciones: ')
display(recomendaciones)
print(f'Recomendaciones por Año ({year_filtrado}): ')
display(recomendaciones_filtradas)

Recomendaciones: 


,title,year
782,Юленька (2009),2009.0
4212,It's in the Blood (2012),2012.0
3292,Dorm (2006),2006.0
2219,Mr. Jones (2013),2013.0
447,Meet Me There (2014),2014.0


Recomendaciones por Año (2010): 


,title,year
4212,It's in the Blood (2012),2012.0
2219,Mr. Jones (2013),2013.0
447,Meet Me There (2014),2014.0
